# Stock Forecasting: Modeling and Temporal Validation

## Prediction contract

At the end of trading day $t$, the model uses completed price and volume
information available through that session to predict:

1. The closing price on the next trading day, $t+1$.
2. The close-to-close return from day $t$ to day $t+1$.

The primary statistical target is the next-day return:

$$
r_{i,t+1}
=
\frac{\mathrm{Close}_{i,t+1}}
{\mathrm{Close}_{i,t}}
- 1
$$

where:

- $i$ represents the stock ticker.
- $t$ represents the current trading day.
- $r_{i,t+1}$ represents the stock's return on the next trading day.

The predicted return is converted into a next-day closing-price forecast using:

$$
\widehat{\mathrm{Close}}_{i,t+1}
=
\mathrm{Close}_{i,t}
\left(1+\widehat{r}_{i,t+1}\right)
$$

All dataset splits are chronological. A random train-test split is not used
because it could allow future market observations to influence model
development and produce an unrealistically optimistic evaluation.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().parent
PRICE_PATH = PROJECT_ROOT / "data" / "raw" / "price.csv"
NEWS_PATH = PROJECT_ROOT / "data" / "raw" / "news.csv"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
prices = (
    pd.read_csv(PRICE_PATH, parse_dates=["date"])
    .assign(
        ticker=lambda df: df["ticker"].str.strip().str.upper()
    )
    .drop_duplicates()
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

news = (
    pd.read_csv(NEWS_PATH, parse_dates=["datetime"])
    .assign(
        ticker=lambda df: df["ticker"].str.strip().str.upper(),
        headline=lambda df: df["headline"].str.strip(),
        summary=lambda df: df["summary"].fillna("").str.strip(),
    )
    .drop_duplicates()
    .sort_values(["ticker", "datetime"])
    .reset_index(drop=True)
)

print(f"Prices: {prices.shape}")
print(f"News:   {news.shape}")

Prices: (1687, 7)
News:   (4439, 4)


Construct the targets

In [3]:
modeling_data = prices.copy()

grouped_close = modeling_data.groupby("ticker")["close"]

modeling_data["current_return"] = grouped_close.pct_change(
    fill_method=None
)

modeling_data["next_close"] = grouped_close.shift(-1)

modeling_data["next_return"] = (
    modeling_data["next_close"] / modeling_data["close"] - 1
)

modeling_data["target_direction"] = (
    modeling_data["next_return"] > 0
).astype(int)

target_check = modeling_data[
    [
        "date",
        "ticker",
        "close",
        "current_return",
        "next_close",
        "next_return",
        "target_direction",
    ]
].head(10)

display(target_check)

,date,ticker,close,current_return,next_close,next_return,target_direction
0,2023-11-13,AAPL,184.800003,NaN,187.440002,0.014286,1
1,2023-11-14,AAPL,187.440002,0.014286,188.009995,0.003041,1
2,2023-11-15,AAPL,188.009995,0.003041,189.710007,0.009042,1
3,2023-11-16,AAPL,189.710007,0.009042,189.690002,-0.000105,0
4,2023-11-17,AAPL,189.690002,-0.000105,191.449997,0.009278,1
5,2023-11-20,AAPL,191.449997,0.009278,190.639999,-0.004231,0
6,2023-11-21,AAPL,190.639999,-0.004231,191.309998,0.003514,1
7,2023-11-22,AAPL,191.309998,0.003514,189.970001,-0.007004,0
8,2023-11-24,AAPL,189.970001,-0.007004,189.789993,-0.000948,0
9,2023-11-27,AAPL,189.789993,-0.000948,190.399994,0.003214,1


In [4]:
print(
    "Rows without next-day targets:",
    modeling_data["next_return"].isna().sum(),
)

display(
    modeling_data[
        modeling_data["next_return"].isna()
    ][["date", "ticker", "close", "next_close"]]
)

Rows without next-day targets: 7


,date,ticker,close,next_close
240,2024-10-28,AAPL,233.399994,NaN
481,2024-10-28,AMZN,188.389999,NaN
722,2024-10-28,GOOGL,166.720001,NaN
963,2024-10-28,META,578.159973,NaN
1204,2024-10-28,MSFT,426.589996,NaN
1445,2024-10-28,NVDA,140.520004,NaN
1686,2024-10-28,TSLA,262.510010,NaN


Remove records without targets

In [5]:
modeling_data = (
    modeling_data
    .dropna(subset=["next_close", "next_return"])
    .reset_index(drop=True)
)

print(f"Modeling rows with targets: {len(modeling_data):,}")

Modeling rows with targets: 1,680


train-val-test split

In [6]:
unique_dates = np.array(
    sorted(modeling_data["date"].unique())
)

train_end_index = int(len(unique_dates) * 0.70)
validation_end_index = int(len(unique_dates) * 0.85)

train_dates = unique_dates[:train_end_index]
validation_dates = unique_dates[
    train_end_index:validation_end_index
]
test_dates = unique_dates[validation_end_index:]

train = modeling_data[
    modeling_data["date"].isin(train_dates)
].copy()

validation = modeling_data[
    modeling_data["date"].isin(validation_dates)
].copy()

test = modeling_data[
    modeling_data["date"].isin(test_dates)
].copy()

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "unique_dates": [
            train["date"].nunique(),
            validation["date"].nunique(),
            test["date"].nunique(),
        ],
        "first_date": [
            train["date"].min(),
            validation["date"].min(),
            test["date"].min(),
        ],
        "last_date": [
            train["date"].max(),
            validation["date"].max(),
            test["date"].max(),
        ],
    }
)

display(split_summary)

,split,rows,unique_dates,first_date,last_date
0,train,1176,168,2023-11-13,2024-07-16
1,validation,252,36,2024-07-17,2024-09-05
2,test,252,36,2024-09-06,2024-10-25


In [7]:
assert train["date"].max() < validation["date"].min()
assert validation["date"].max() < test["date"].min()

assert set(train["date"]).isdisjoint(validation["date"])
assert set(train["date"]).isdisjoint(test["date"])
assert set(validation["date"]).isdisjoint(test["date"])

print("Chronological split validation passed.")

Chronological split validation passed.


Naive random-walk baseline:
Assumes tomorrow's closing price equals today's
$$
\widehat{\mathrm{Close}}_{t+1}
=
\mathrm{Close}_{t}
$$

In [8]:
def regression_metrics(
    actual_price: pd.Series,
    predicted_price: pd.Series,
    actual_return: pd.Series,
    predicted_return: pd.Series,
) -> dict:
    return {
        "price_mae": mean_absolute_error(
            actual_price,
            predicted_price,
        ),
        "price_rmse": np.sqrt(
            mean_squared_error(
                actual_price,
                predicted_price,
            )
        ),
        "return_mae": mean_absolute_error(
            actual_return,
            predicted_return,
        ),
        "return_rmse": np.sqrt(
            mean_squared_error(
                actual_return,
                predicted_return,
            )
        ),
    }

In [9]:
def evaluate_random_walk(
    data: pd.DataFrame,
    split_name: str,
) -> dict:
    predicted_price = data["close"]
    predicted_return = np.zeros(len(data))

    metrics = regression_metrics(
        actual_price=data["next_close"],
        predicted_price=predicted_price,
        actual_return=data["next_return"],
        predicted_return=predicted_return,
    )

    return {
        "model": "Random walk",
        "split": split_name,
        **metrics,
    }


baseline_results = pd.DataFrame(
    [
        evaluate_random_walk(train, "train"),
        evaluate_random_walk(validation, "validation"),
        evaluate_random_walk(test, "test"),
    ]
)

display(
    baseline_results.style.format(
        {
            "price_mae": "${:,.3f}",
            "price_rmse": "${:,.3f}",
            "return_mae": "{:.3%}",
            "return_rmse": "{:.3%}",
        }
    )
)

,model,split,price_mae,price_rmse,return_mae,return_rmse
0,Random walk,train,$3.274,$5.526,1.497%,2.253%
1,Random walk,validation,$4.563,$6.574,1.999%,2.871%
2,Random walk,test,$3.303,$5.463,1.340%,2.270%


In [10]:
ticker_baseline_results = []

for ticker, group in test.groupby("ticker"):
    metrics = regression_metrics(
        actual_price=group["next_close"],
        predicted_price=group["close"],
        actual_return=group["next_return"],
        predicted_return=np.zeros(len(group)),
    )

    ticker_baseline_results.append(
        {
            "ticker": ticker,
            **metrics,
        }
    )

ticker_baseline_results = pd.DataFrame(
    ticker_baseline_results
).sort_values("return_rmse", ascending=False)

display(
    ticker_baseline_results.style.format(
        {
            "price_mae": "${:,.3f}",
            "price_rmse": "${:,.3f}",
            "return_mae": "{:.3%}",
            "return_rmse": "{:.3%}",
        }
    )
)

,ticker,price_mae,price_rmse,return_mae,return_rmse
6,TSLA,$6.101,$10.306,2.613%,4.618%
5,NVDA,$2.525,$3.233,2.057%,2.670%
1,AMZN,$1.991,$2.487,1.078%,1.354%
0,AAPL,$2.218,$3.057,0.978%,1.352%
3,META,$5.531,$7.462,0.986%,1.338%
2,GOOGL,$1.410,$1.717,0.877%,1.073%
4,MSFT,$3.348,$4.254,0.794%,1.011%


### Random-walk baseline results

The random-walk baseline assumes that the next closing price will equal the
current closing price, corresponding to a predicted return of zero.

On the held-out test period, this baseline achieved a price MAE of approximately
$3.30 and a return MAE of 1.34%. These scores establish the minimum benchmark
that more sophisticated models must improve upon.

Performance differs across the three chronological periods. The validation
period has higher errors than both the training and test periods, suggesting
that it represents a comparatively difficult or volatile market regime. This
variation demonstrates why evaluation across multiple chronological periods is
more informative than relying on one random split.

Ticker-level results are consistent with the exploratory volatility analysis.
Tesla is the most difficult stock to forecast, with a test return RMSE of 4.62%,
followed by NVIDIA at 2.67%. Microsoft and Google have the lowest return RMSE
values, reflecting their comparatively narrower daily-return distributions.

Price errors are influenced by the scale of each stock's price and should not
be compared across tickers in isolation. Return-based metrics provide a more
comparable assessment of forecasting difficulty.

## Price and volume feature engineering

Features are calculated using information available through the end of trading
day \(t\). The target remains the return on trading day \(t+1\).

Rolling calculations are performed independently for each ticker to prevent
information from crossing company boundaries.

In [11]:
features = prices.sort_values(["ticker", "date"]).copy()

close_group = features.groupby("ticker")["close"]
volume_group = features.groupby("ticker")["volume"]

# Returns and momentum
features["return_1d"] = close_group.pct_change(fill_method=None)

for window in [2, 5, 10, 20]:
    features[f"return_{window}d"] = (
        features["close"]
        / close_group.shift(window)
        - 1
    )

# Intraday behavior
features["intraday_return"] = (
    features["close"] / features["open"] - 1
)

features["intraday_range"] = (
    features["high"] - features["low"]
) / features["open"]

features["close_position"] = (
    features["close"] - features["low"]
) / (features["high"] - features["low"]).replace(0, np.nan)

# Rolling statistics
for window in [5, 10, 20]:
    features[f"mean_return_{window}d"] = (
        features
        .groupby("ticker")["return_1d"]
        .transform(
            lambda values: values.rolling(window).mean()
        )
    )

    features[f"volatility_{window}d"] = (
        features
        .groupby("ticker")["return_1d"]
        .transform(
            lambda values: values.rolling(window).std()
        )
    )

    moving_average = (
        features
        .groupby("ticker")["close"]
        .transform(
            lambda values: values.rolling(window).mean()
        )
    )

    features[f"close_to_sma_{window}d"] = (
        features["close"] / moving_average - 1
    )

# Volume features
features["volume_change_1d"] = (
    volume_group.pct_change(fill_method=None)
)

features["volume_mean_20d"] = (
    features
    .groupby("ticker")["volume"]
    .transform(
        lambda values: values.rolling(20).mean()
    )
)

features["relative_volume_20d"] = (
    features["volume"] / features["volume_mean_20d"]
)

# Cross-sectional market context known at the end of day t
features["market_return_1d"] = (
    features
    .groupby("date")["return_1d"]
    .transform("mean")
)

features["relative_return_1d"] = (
    features["return_1d"]
    - features["market_return_1d"]
)

# Calendar variables
features["day_of_week"] = features["date"].dt.dayofweek
features["month"] = features["date"].dt.month

# Next-trading-day targets
features["next_close"] = close_group.shift(-1)

features["next_return"] = (
    features["next_close"] / features["close"] - 1
)

features.replace([np.inf, -np.inf], np.nan, inplace=True)

,date,ticker,open,high,low,close,volume,return_1d,return_2d,return_5d,return_10d,return_20d,intraday_return,intraday_range,close_position,mean_return_5d,volatility_5d,close_to_sma_5d,mean_return_10d,volatility_10d,close_to_sma_10d,mean_return_20d,volatility_20d,close_to_sma_20d,volume_change_1d,volume_mean_20d,relative_volume_20d,market_return_1d,relative_return_1d,day_of_week,month,next_close,next_return
0,2023-11-13,AAPL,185.820007,186.029999,184.210007,184.800003,43627500,NaN,NaN,NaN,NaN,NaN,-0.005489,0.009794,0.324175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,11,187.440002,0.014286
1,2023-11-14,AAPL,187.699997,188.110001,186.300003,187.440002,60108400,0.014286,NaN,NaN,NaN,NaN,-0.001385,0.009643,0.629835,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.377764,NaN,NaN,0.023193,-0.008907,1,11,188.009995,0.003041
2,2023-11-15,AAPL,187.850006,189.500000,187.779999,188.009995,53790500,0.003041,0.017370,NaN,NaN,NaN,0.000852,0.009156,0.133718,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.105108,NaN,NaN,-0.001747,0.004788,2,11,189.710007,0.009042
3,2023-11-16,AAPL,189.570007,190.960007,188.649994,189.710007,54412900,0.009042,0.012111,NaN,NaN,NaN,0.000739,0.012186,0.458877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.011571,NaN,NaN,0.002810,0.006232,3,11,189.690002,-0.000105
4,2023-11-17,AAPL,190.250000,190.380005,188.570007,189.690002,50922700,-0.000105,0.008936,NaN,NaN,NaN,-0.002943,0.009514,0.618783,NaN,NaN,0.009365,NaN,NaN,NaN,NaN,NaN,NaN,-0.064143,NaN,NaN,-0.001483,0.001377,4,11,191.449997,0.009278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1682,2024-10-22,TSLA,217.309998,218.220001,215.259995,217.970001,43268700,-0.004021,-0.012370,-0.007287,-0.108507,-0.142762,0.003037,0.013621,0.915541,-0.001447,0.006019,-0.008993,-0.011052,0.027845,-0.025218,-0.007321,0.026806,-0.082720,-0.085789,"71,944,555.000000",0.601417,0.005019,-0.009040,1,10,213.649994,-0.019819
1683,2024-10-23,TSLA,217.130005,218.720001,212.110001,213.649994,80938900,-0.019819,-0.023761,-0.034699,-0.113669,-0.168742,-0.016027,0.030443,0.232979,-0.007014,0.007713,-0.021803,-0.011622,0.027973,-0.032685,-0.008852,0.026590,-0.092620,0.870611,"72,739,785.000000",1.112718,-0.021198,0.001379,2,10,260.480011,0.219190
1684,2024-10-24,TSLA,244.679993,262.119995,242.649994,260.480011,204491900,0.219190,0.195027,0.179230,0.090924,0.024624,0.064574,0.079573,0.915769,0.037221,0.101977,0.150886,0.011243,0.078234,0.167862,0.002652,0.057485,0.104801,1.526497,"79,607,270.000000",2.568759,0.034385,0.184806,3,10,269.190002,0.033438
1685,2024-10-25,TSLA,256.010010,269.489990,255.320007,269.190002,161611900,0.033438,0.259958,0.219710,0.235950,0.033518,0.051482,0.055349,0.978829,0.044081,0.099907,0.140500,0.023369,0.070152,0.179732,0.003096,0.057697,0.139634,-0.209690,"84,138,460.000000",1.920785,0.012324,0.021114,4,10,262.510010,-0.024815


Select Modeling Columns

In [13]:
numeric_features = [
    "return_1d",
    "return_2d",
    "return_5d",
    "return_10d",
    "return_20d",
    "intraday_return",
    "intraday_range",
    "close_position",
    "mean_return_5d",
    "mean_return_10d",
    "mean_return_20d",
    "volatility_5d",
    "volatility_10d",
    "volatility_20d",
    "close_to_sma_5d",
    "close_to_sma_10d",
    "close_to_sma_20d",
    "volume_change_1d",
    "relative_volume_20d",
    "market_return_1d",
    "relative_return_1d",
    "day_of_week",
    "month",
]

categorical_features = ["ticker"]

required_columns = (
    numeric_features
    + categorical_features
    + ["date", "close", "next_close", "next_return"]
)

model_data = (
    features[required_columns]
    .dropna()
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

print(f"Rows before feature filtering: {len(features):,}")
print(f"Complete modeling rows: {len(model_data):,}")
print(f"Unique modeling dates: {model_data['date'].nunique()}")

Rows before feature filtering: 1,687
Complete modeling rows: 1,540
Unique modeling dates: 220


Train-Val-Test Split

In [14]:
model_dates = np.array(sorted(model_data["date"].unique()))

train_end = int(len(model_dates) * 0.70)
validation_end = int(len(model_dates) * 0.85)

train_dates = model_dates[:train_end]
validation_dates = model_dates[train_end:validation_end]
test_dates = model_dates[validation_end:]

train = model_data[model_data["date"].isin(train_dates)].copy()
validation = model_data[
    model_data["date"].isin(validation_dates)
].copy()
test = model_data[model_data["date"].isin(test_dates)].copy()

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "dates": [
            train["date"].nunique(),
            validation["date"].nunique(),
            test["date"].nunique(),
        ],
        "start": [
            train["date"].min(),
            validation["date"].min(),
            test["date"].min(),
        ],
        "end": [
            train["date"].max(),
            validation["date"].max(),
            test["date"].max(),
        ],
    }
)

display(split_summary)

,split,rows,dates,start,end
0,train,1078,154,2023-12-12,2024-07-24
1,validation,231,33,2024-07-25,2024-09-10
2,test,231,33,2024-09-11,2024-10-25


Train Ridge Regression Model

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
        (
            "ticker",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
    ]
)

ridge_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]
)

X_train = train[numeric_features + categorical_features]
y_train = train["next_return"]

ridge_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](24,)","['return_1d','return_2d','return_5d',...,'day_of_week','month','ticker']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,24
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('ticker', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of

Evaluate Ridge Model and Compare with Baseline

In [17]:
def evaluate_return_model(
    model,
    data: pd.DataFrame,
    model_name: str,
    split_name: str,
) -> dict:
    X = data[numeric_features + categorical_features]

    predicted_return = model.predict(X)
    predicted_price = data["close"] * (1 + predicted_return)

    actual_direction = data["next_return"] > 0
    predicted_direction = predicted_return > 0

    return {
        "model": model_name,
        "split": split_name,
        "price_mae": mean_absolute_error(
            data["next_close"],
            predicted_price,
        ),
        "price_rmse": np.sqrt(
            mean_squared_error(
                data["next_close"],
                predicted_price,
            )
        ),
        "return_mae": mean_absolute_error(
            data["next_return"],
            predicted_return,
        ),
        "return_rmse": np.sqrt(
            mean_squared_error(
                data["next_return"],
                predicted_return,
            )
        ),
        "direction_accuracy": (
            actual_direction == predicted_direction
        ).mean(),
    }

In [18]:
comparison_results = []

for split_name, split_data in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    # Random-walk baseline on the exact same rows
    baseline = regression_metrics(
        actual_price=split_data["next_close"],
        predicted_price=split_data["close"],
        actual_return=split_data["next_return"],
        predicted_return=np.zeros(len(split_data)),
    )

    comparison_results.append(
        {
            "model": "Random walk",
            "split": split_name,
            **baseline,
            "direction_accuracy": np.nan,
        }
    )

    comparison_results.append(
        evaluate_return_model(
            ridge_model,
            split_data,
            "Ridge",
            split_name,
        )
    )

comparison_results = pd.DataFrame(comparison_results)

display(
    comparison_results.style.format(
        {
            "price_mae": "${:,.3f}",
            "price_rmse": "${:,.3f}",
            "return_mae": "{:.3%}",
            "return_rmse": "{:.3%}",
            "direction_accuracy": "{:.2%}",
        }
    )
)

,model,split,price_mae,price_rmse,return_mae,return_rmse,direction_accuracy
0,Random walk,train,$3.468,$5.890,1.557%,2.370%,nan%
1,Ridge,train,$3.432,$5.839,1.527%,2.314%,57.70%
2,Random walk,validation,$4.310,$5.982,1.951%,2.784%,nan%
3,Ridge,validation,$4.526,$6.251,2.052%,2.948%,43.29%
4,Random walk,test,$3.233,$5.498,1.285%,2.232%,nan%
5,Ridge,test,$3.238,$5.542,1.300%,2.275%,48.92%


### Ridge regression results

Ridge regression produces a small improvement over the random-walk baseline on
the training period but fails to maintain that improvement on unseen dates.

On the validation period, Ridge increases return MAE from 1.95% to 2.05% and
reduces directional accuracy to 43.29%. Test performance is closer to the
baseline, but Ridge remains slightly worse in both price and return metrics.
Its test directional accuracy of 48.92% is also below 50%.

The difference between training and unseen-period performance indicates that
the linear relationships identified in historical returns, volatility, volume,
and market features are not stable across the full sample. This may reflect
nonlinear behavior, changing market regimes, or a low signal-to-noise ratio in
next-day returns.

The result reinforces the importance of comparing every model against a simple
random-walk benchmark. A more complex model should only be preferred if it
produces consistent improvements on chronological validation data.

Try Tree models-> XGBoost

In [19]:
from xgboost import XGBRegressor

In [20]:
X_train_processed = preprocessor.fit_transform(
    train[numeric_features + categorical_features]
)

X_validation_processed = preprocessor.transform(
    validation[numeric_features + categorical_features]
)

X_test_processed = preprocessor.transform(
    test[numeric_features + categorical_features]
)

y_train = train["next_return"]
y_validation = validation["next_return"]
y_test = test["next_return"]

print("Training matrix:", X_train_processed.shape)
print("Validation matrix:", X_validation_processed.shape)
print("Test matrix:", X_test_processed.shape)

Training matrix: (1078, 30)
Validation matrix: (231, 30)
Test matrix: (231, 30)


In [21]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.03,
    max_depth=2,
    min_child_weight=10,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.01,
    reg_lambda=5.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=40,
)

xgb_model.fit(
    X_train_processed,
    y_train,
    eval_set=[
        (X_validation_processed, y_validation)
    ],
    verbose=False,
)

print("Best iteration:", xgb_model.best_iteration)

Best iteration: 2


Evaluate XGBoost

In [22]:
def evaluate_processed_model(
    model,
    X,
    data: pd.DataFrame,
    model_name: str,
    split_name: str,
) -> dict:
    predicted_return = model.predict(X)
    predicted_price = data["close"] * (1 + predicted_return)

    return {
        "model": model_name,
        "split": split_name,
        "price_mae": mean_absolute_error(
            data["next_close"],
            predicted_price,
        ),
        "price_rmse": np.sqrt(
            mean_squared_error(
                data["next_close"],
                predicted_price,
            )
        ),
        "return_mae": mean_absolute_error(
            data["next_return"],
            predicted_return,
        ),
        "return_rmse": np.sqrt(
            mean_squared_error(
                data["next_return"],
                predicted_return,
            )
        ),
        "direction_accuracy": (
            (data["next_return"].to_numpy() > 0)
            == (predicted_return > 0)
        ).mean(),
    }

In [23]:
xgb_results = pd.DataFrame(
    [
        evaluate_processed_model(
            xgb_model,
            X_train_processed,
            train,
            "XGBoost",
            "train",
        ),
        evaluate_processed_model(
            xgb_model,
            X_validation_processed,
            validation,
            "XGBoost",
            "validation",
        ),
        evaluate_processed_model(
            xgb_model,
            X_test_processed,
            test,
            "XGBoost",
            "test",
        ),
    ]
)

all_model_results = pd.concat(
    [
        comparison_results,
        xgb_results,
    ],
    ignore_index=True,
)

display(
    all_model_results.style.format(
        {
            "price_mae": "${:,.3f}",
            "price_rmse": "${:,.3f}",
            "return_mae": "{:.3%}",
            "return_rmse": "{:.3%}",
            "direction_accuracy": "{:.2%}",
        }
    )
)

,model,split,price_mae,price_rmse,return_mae,return_rmse,direction_accuracy
0,Random walk,train,$3.468,$5.890,1.557%,2.370%,nan%
1,Ridge,train,$3.432,$5.839,1.527%,2.314%,57.70%
2,Random walk,validation,$4.310,$5.982,1.951%,2.784%,nan%
3,Ridge,validation,$4.526,$6.251,2.052%,2.948%,43.29%
4,Random walk,test,$3.233,$5.498,1.285%,2.232%,nan%
5,Ridge,test,$3.238,$5.542,1.300%,2.275%,48.92%
6,XGBoost,train,$3.455,$5.884,1.547%,2.357%,54.17%
7,XGBoost,validation,$4.287,$5.968,1.937%,2.783%,54.98%
8,XGBoost,test,$3.199,$5.453,1.266%,2.215%,58.01%


In [24]:
training_positive_rate = (
    train["next_return"] > 0
).mean()

majority_prediction = training_positive_rate >= 0.5

direction_baselines = []

for split_name, split_data in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    actual_direction = split_data["next_return"] > 0

    majority_accuracy = (
        actual_direction == majority_prediction
    ).mean()

    momentum_accuracy = (
        actual_direction
        == (split_data["return_1d"] > 0)
    ).mean()

    direction_baselines.extend(
        [
            {
                "model": "Training-majority direction",
                "split": split_name,
                "direction_accuracy": majority_accuracy,
            },
            {
                "model": "Previous-day direction",
                "split": split_name,
                "direction_accuracy": momentum_accuracy,
            },
        ]
    )

direction_baselines = pd.DataFrame(direction_baselines)

display(
    direction_baselines.style.format(
        {"direction_accuracy": "{:.2%}"}
    )
)

,model,split,direction_accuracy
0,Training-majority direction,train,54.17%
1,Previous-day direction,train,51.95%
2,Training-majority direction,validation,54.98%
3,Previous-day direction,validation,42.86%
4,Training-majority direction,test,58.01%
5,Previous-day direction,test,51.08%


In [25]:
for split_name, X_processed in [
    ("train", X_train_processed),
    ("validation", X_validation_processed),
    ("test", X_test_processed),
]:
    predictions = xgb_model.predict(X_processed)

    print(
        split_name,
        {
            "minimum_prediction": predictions.min(),
            "maximum_prediction": predictions.max(),
            "mean_prediction": predictions.mean(),
            "predicted_positive_share": (
                predictions > 0
            ).mean(),
        },
    )

train {'minimum_prediction': np.float32(0.001671503), 'maximum_prediction': np.float32(0.0034781836), 'mean_prediction': np.float32(0.0018658001), 'predicted_positive_share': np.float64(1.0)}
validation {'minimum_prediction': np.float32(0.001693827), 'maximum_prediction': np.float32(0.0033132355), 'mean_prediction': np.float32(0.0019102592), 'predicted_positive_share': np.float64(1.0)}
test {'minimum_prediction': np.float32(0.001671503), 'maximum_prediction': np.float32(0.0033132355), 'mean_prediction': np.float32(0.0018325817), 'predicted_positive_share': np.float64(1.0)}


In [26]:
global_mean_return = train["next_return"].mean()

ticker_mean_returns = (
    train
    .groupby("ticker")["next_return"]
    .mean()
    .to_dict()
)

print(f"Training global mean return: {global_mean_return:.4%}")
display(
    pd.Series(
        ticker_mean_returns,
        name="training_mean_return",
    ).to_frame().style.format("{:.4%}")
)

Training global mean return: 0.1892%


,training_mean_return
AAPL,0.0832%
AMZN,0.1407%
GOOGL,0.1667%
META,0.2304%
MSFT,0.0797%
NVDA,0.6089%
TSLA,0.0151%


In [27]:
def evaluate_constant_predictions(
    data: pd.DataFrame,
    predicted_return: np.ndarray,
    model_name: str,
    split_name: str,
) -> dict:
    predicted_price = (
        data["close"].to_numpy()
        * (1 + predicted_return)
    )

    return {
        "model": model_name,
        "split": split_name,
        "price_mae": mean_absolute_error(
            data["next_close"],
            predicted_price,
        ),
        "price_rmse": np.sqrt(
            mean_squared_error(
                data["next_close"],
                predicted_price,
            )
        ),
        "return_mae": mean_absolute_error(
            data["next_return"],
            predicted_return,
        ),
        "return_rmse": np.sqrt(
            mean_squared_error(
                data["next_return"],
                predicted_return,
            )
        ),
        "direction_accuracy": (
            (data["next_return"].to_numpy() > 0)
            == (predicted_return > 0)
        ).mean(),
    }

In [28]:
constant_baseline_results = []

for split_name, split_data in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    global_predictions = np.full(
        len(split_data),
        global_mean_return,
    )

    ticker_predictions = (
        split_data["ticker"]
        .map(ticker_mean_returns)
        .to_numpy()
    )

    constant_baseline_results.extend(
        [
            evaluate_constant_predictions(
                split_data,
                global_predictions,
                "Global historical mean",
                split_name,
            ),
            evaluate_constant_predictions(
                split_data,
                ticker_predictions,
                "Ticker historical mean",
                split_name,
            ),
        ]
    )

constant_baseline_results = pd.DataFrame(
    constant_baseline_results
)

expanded_results = pd.concat(
    [
        comparison_results,
        xgb_results,
        constant_baseline_results,
    ],
    ignore_index=True,
)

display(
    expanded_results.style.format(
        {
            "price_mae": "${:,.3f}",
            "price_rmse": "${:,.3f}",
            "return_mae": "{:.3%}",
            "return_rmse": "{:.3%}",
            "direction_accuracy": "{:.2%}",
        }
    )
)

,model,split,price_mae,price_rmse,return_mae,return_rmse,direction_accuracy
0,Random walk,train,$3.468,$5.890,1.557%,2.370%,nan%
1,Ridge,train,$3.432,$5.839,1.527%,2.314%,57.70%
2,Random walk,validation,$4.310,$5.982,1.951%,2.784%,nan%
3,Ridge,validation,$4.526,$6.251,2.052%,2.948%,43.29%
4,Random walk,test,$3.233,$5.498,1.285%,2.232%,nan%
5,Ridge,test,$3.238,$5.542,1.300%,2.275%,48.92%
6,XGBoost,train,$3.455,$5.884,1.547%,2.357%,54.17%
7,XGBoost,validation,$4.287,$5.968,1.937%,2.783%,54.98%
8,XGBoost,test,$3.199,$5.453,1.266%,2.215%,58.01%
9,Global historical mean,train,$3.462,$5.895,1.551%,2.363%,54.17%


### XGBoost results

XGBoost produces slightly lower price and return errors than the random-walk
baseline on both validation and test data. However, its directional accuracy
exactly matches the training-majority baseline in every split.

This result suggests that the model may predict a positive return for nearly
every observation rather than distinguishing between positive and negative
days. Its apparent directional performance therefore cannot be interpreted as
evidence of successful market-direction forecasting.

The small error improvement may instead come from estimating the sample's
positive average return more effectively than the zero-return random-walk
baseline. Constant global and ticker-specific historical-mean forecasts are
introduced to determine whether XGBoost provides incremental predictive value
beyond this simple drift estimate.

### Comparison with historical-drift baselines

The constant-return baselines demonstrate that XGBoost provides almost no
incremental improvement beyond estimating positive historical drift.

On validation and test data, the global historical-mean forecast produces
nearly identical errors to XGBoost. The ticker-specific historical-mean model
slightly outperforms XGBoost on the test period, achieving a return MAE of
1.261% compared with 1.266%.

All three approaches achieve identical directional accuracy because they
predict positive returns for nearly every observation. Their directional
accuracy therefore represents the proportion of positive observations in each
split rather than successful discrimination between positive and negative
market movements.

These results indicate that the structured price and volume features have not
yet provided stable incremental signal beyond historical drift. The news
ablation will test whether textual market information improves this result.